# FoodSQL-Agent: QLoRA training and evaluation on Kaggle

Before running: Settings -> Accelerator = **GPU T4** (not P100), Internet = **On**,
and add the private dataset that contains `kaggle_bundle.zip` (src, datasets, db).
Run the cells from top to bottom.

In [ ]:
!nvidia-smi

## 1. Unpack the project

In [ ]:
import glob, os, shutil, subprocess

WORK = "/kaggle/working/foodsql-agent"
os.makedirs(WORK, exist_ok=True)

# Kaggle usually extracts uploaded zip files itself, so try the extracted folder first
hits = glob.glob("/kaggle/input/**/src/agent/schema.py", recursive=True)
if hits:
    root = hits[0].split("/src/agent/schema.py")[0]
    shutil.copytree(root, WORK, dirs_exist_ok=True)
else:
    bundle = glob.glob("/kaggle/input/**/kaggle_bundle.zip", recursive=True)[0]
    subprocess.run(["python", "-m", "zipfile", "-e", bundle, WORK], check=True)

os.chdir(WORK)
print(os.getcwd(), sorted(os.listdir(".")))

## 2. Libraries (only bitsandbytes and peft are updated)

In [ ]:
!pip install -q -U bitsandbytes peft
import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 3. Smoke test (about 3 minutes). Only continue if this works

In [ ]:
!python src/evaluation/run_eval.py --model hf --model-id Qwen/Qwen2.5-Coder-7B-Instruct --limit 5 --tag smoke_base

## 4. Fine-tune Qwen2.5-Coder-7B-Instruct with QLoRA (roughly 1 to 1.5 hours)

In [ ]:
!python src/finetune/train_qlora.py --model-id Qwen/Qwen2.5-Coder-7B-Instruct --epochs 2 --out outputs/qwen7b-lora

## 5. Evaluate: base model (before fine-tuning), with and without the self-correction loop

In [ ]:
!python src/evaluation/run_eval.py --model hf --model-id Qwen/Qwen2.5-Coder-7B-Instruct --no-loop
!python src/evaluation/run_eval.py --model hf --model-id Qwen/Qwen2.5-Coder-7B-Instruct

## 6. Evaluate: fine-tuned model, with and without the self-correction loop

In [ ]:
!python src/evaluation/run_eval.py --model hf --model-id Qwen/Qwen2.5-Coder-7B-Instruct --adapter outputs/qwen7b-lora --no-loop
!python src/evaluation/run_eval.py --model hf --model-id Qwen/Qwen2.5-Coder-7B-Instruct --adapter outputs/qwen7b-lora

## 7. Pack the results and the adapter for download

In [ ]:
!zip -qr results_and_adapter.zip results outputs
!ls -lh results_and_adapter.zip